# HR Data Cleaning 

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Load Dataset

In [2]:
df=pd.read_csv("messy_HR_data.csv")
df_before= df.copy()
df.head()

,Name,Age,Salary,Gender,Department,Position,Joining Date,Performance Score,Email,Phone Number
0,grace,25,50000,Male,HR,Manager,"April 5, 2018",D,email@example.com,NaN
1,david,NaN,65000,Female,Finance,Director,2020/02/20,F,user@domain.com,123-456-7890
2,hannah,35,SIXTY THOUSAND,Female,Sales,Director,01/15/2020,C,email@example.com,098-765-4321
3,eve,NaN,50000,Female,IT,Manager,"April 5, 2018",A,name@company.org,
4,grace,NaN,NAN,Female,Finance,Manager,01/15/2020,F,name@company.org,098-765-4321


## Inspection

In [3]:
df.shape


(1000, 10)

In [4]:
df.columns.tolist()

['Name',
 'Age',
 'Salary',
 'Gender',
 'Department',
 'Position',
 'Joining Date',
 'Performance Score',
 'Email',
 'Phone Number']

In [5]:
df.isnull().sum().sum()


np.int64(734)

In [6]:
df.isnull().sum().sort_values(ascending=False)

Email                390
Phone Number         185
Age                  159
Name                   0
Salary                 0
Gender                 0
Position               0
Department             0
Performance Score      0
Joining Date           0
dtype: int64

In [7]:
df.duplicated().sum()


np.int64(0)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               1000 non-null   str  
 1   Age                841 non-null    str  
 2   Salary             1000 non-null   str  
 3   Gender             1000 non-null   str  
 4   Department         1000 non-null   str  
 5   Position           1000 non-null   str  
 6   Joining Date       1000 non-null   str  
 7   Performance Score  1000 non-null   str  
 8   Email              610 non-null    str  
 9   Phone Number       815 non-null    str  
dtypes: str(10)
memory usage: 138.7 KB


In [9]:
df.describe()

,Name,Age,Salary,Gender,Department,Position,Joining Date,Performance Score,Email,Phone Number
count,1000,841,1000,1000,1000,1000,1000,1000,610,815
unique,10,5,6,3,5,5,5,5,3,4
top,alice,thirty,65000,Male,Finance,Assistant,2020/02/20,B,user@domain.com,123-456-7890
freq,118,176,184,355,218,214,232,225,213,236


In [10]:
df['Salary'].unique()


<ArrowStringArray>
['50000', '65000', 'SIXTY THOUSAND', ' NAN ', '70000', '55000']
Length: 6, dtype: str

In [11]:
df['Age'].unique()

<ArrowStringArray>
['25', nan, '35', '40', 'thirty', '50']
Length: 6, dtype: str

### Checking how many values are in word form in Salary and Age column

In [12]:
(~df['Age'].str.isdigit()).sum()

np.int64(335)

In [13]:
(~df['Salary'].str.isdigit()).sum()

np.int64(310)

In [14]:
df['Gender'].unique()

<ArrowStringArray>
['Male', 'Female', 'Other']
Length: 3, dtype: str

In [15]:
df['Department'].unique()

<ArrowStringArray>
['HR', 'Finance', 'Sales', 'IT', 'Marketing']
Length: 5, dtype: str

In [16]:
df['Joining Date'].unique()

<ArrowStringArray>
['April 5, 2018', '2020/02/20', '01/15/2020', '03-25-2019', '2019.12.01']
Length: 5, dtype: str

In [17]:
df['Email'].unique()

<ArrowStringArray>
['email@example.com', 'user@domain.com', 'name@company.org', nan]
Length: 4, dtype: str

In [18]:
df['Phone Number'].unique()

<ArrowStringArray>
[nan, '123-456-7890', '098-765-4321', ' ', '555-555-5555']
Length: 5, dtype: str

In [35]:
df['Performance Score'].unique()

<ArrowStringArray>
['D', 'F', 'C', 'A', 'B']
Length: 5, dtype: str

## 📋 Dataset Inspection Summary

**Dataset:** HR Dataset | **Shape:** 1000 rows × 10 columns

---

### 🔍 Problems Found

| Column | Issue |
|---|---|
| Age | str dtype, 335 word-form values |
| Salary | str dtype, 310 word-form values |
| Joining Date | mixed date formats |
| Phone | nulls + hidden nulls (blank spaces) |
| Email | nulls |
| All Columns | str dtype — no proper types assigned |

---

### ✅ Clean Areas
- Zero duplicates
- Gender: consistent (Male/Female/Other)
- Name, Department, Position, Performance Score: inspected, no major issues found

---

### 🧹 Cleaning Plan
1. Convert word-form values → numeric (Age, Salary)
2. Handle outliers in Age
3. Fix dtypes of all columns
4. Standardize Joining Date formats
5. Handle nulls and hidden nulls (Phone, Email)

## Data Cleaning

In [19]:
pip install word2number

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Converting words to numbers in Salary and Age column

In [20]:
from word2number import w2n

In [21]:
df['Age']=df['Age'].apply(lambda x: w2n.word_to_num(x) if isinstance(x,str) else x)

In [22]:
df['Age'].unique()

array([25., nan, 35., 40., 30., 50.])

In [23]:
df['Age'].isnull().sum()

np.int64(159)

In [26]:
def convert_salary(x):
    if isinstance(x, str):
        if x.strip().isdigit():
            return int(x)
        else:
            try:
                return w2n.word_to_num(x)
            except:
                return None
    return x

In [27]:
df['Salary'] = df['Salary'].apply(convert_salary)

In [28]:
df['Salary'].unique()


array([50000., 65000., 60000.,    nan, 70000., 55000.])

In [29]:
df['Salary'].isna().sum()

np.int64(167)

In [30]:
df_before['Salary'].isna().sum()

np.int64(0)

In [33]:
df_before[df['Salary'].isna()]['Salary'].unique()

<ArrowStringArray>
[' NAN ']
Length: 1, dtype: str

In [24]:
(df_before['Salary'] == ' NAN ').sum()

np.int64(167)

- In original dataset Salary is in str dtype so pandas cannot recognize nulls because 'NAN' is treated as normal string so nulls are `0`.
- After conversion NAN strings are converted to original NAN so nulls are `167`.   

## Datatype Fixing

- Int in pandas can't hold NAN so if i change dtype of Age and Salary to Int conversion fails for NAN 
- Age and Salary dtypes will fixed after handling Nulls

In [34]:
df.dtypes

Name                     str
Age                  float64
Salary               float64
Gender                   str
Department               str
Position                 str
Joining Date             str
Performance Score        str
Email                    str
Phone Number             str
dtype: object

In [46]:
df['Gender'] = df['Gender'].astype('category')
df['Department'] = df['Department'].astype('category')
df['Performance Score'] = df['Performance Score'].astype('category')
df['Position'] = df['Position'].astype('category')

In [47]:
df['Joining Date']=  pd.to_datetime(df['Joining Date'],  format='mixed'   , dayfirst=True, errors='coerce')

In [ ]:
df_before['Joining Date'].isnull().sum()

np.int64(0)

In [48]:
df['Joining Date'].isnull().sum()

np.int64(0)

In [49]:
df.dtypes

Name                            str
Age                         float64
Salary                      float64
Gender                     category
Department                 category
Position                   category
Joining Date         datetime64[us]
Performance Score          category
Email                           str
Phone Number                    str
dtype: object

## Handling Null Values

In [54]:
df.isnull().sum().sort_values(ascending=False)

Email                390
Phone Number         185
Salary               167
Age                  159
Name                   0
Gender                 0
Position               0
Department             0
Performance Score      0
Joining Date           0
dtype: int64

In [53]:
(df.isnull().sum())*100/(len(df))

Name                  0.0
Age                  15.9
Salary               16.7
Gender                0.0
Department            0.0
Position              0.0
Joining Date          0.0
Performance Score     0.0
Email                39.0
Phone Number         18.5
dtype: float64

In [57]:
df['Salary'] = df['Salary'].fillna(df.groupby('Department')['Salary'].transform('mean'))

In [58]:
df['Salary'].isnull().sum()

np.int64(0)

- check for outliers in Age

In [59]:
df['Age'].describe()

count    841.000000
mean      35.802616
std        8.551889
min       25.000000
25%       30.000000
50%       35.000000
75%       40.000000
max       50.000000
Name: Age, dtype: float64

In [60]:
df['Age'] = df['Age'].fillna(df.groupby('Position')['Age'].transform('mean'))

In [61]:
df['Age'].isna().sum()

np.int64(0)

- Converting Age and Salary type to Int.

In [62]:
df['Age']= df['Age'].astype(int)
df['Salary']= df['Salary'].astype(int)

In [63]:
df.dtypes

Name                            str
Age                           int64
Salary                        int64
Gender                     category
Department                 category
Position                   category
Joining Date         datetime64[us]
Performance Score          category
Email                           str
Phone Number                    str
dtype: object

In [64]:
df.isnull().sum()

Name                   0
Age                    0
Salary                 0
Gender                 0
Department             0
Position               0
Joining Date           0
Performance Score      0
Email                390
Phone Number         185
dtype: int64

- Email and phone numbers are contact fields , they can't be guessed so i keep them as NANs

## Standardize name , email , phone

In [65]:
df['Name'] = df['Name'].str.strip().str.title()
df['Email'] = df['Email'].str.strip().str.lower()
df['Phone Number'] = df['Phone Number'].str.strip()

In [66]:
df[['Name', 'Email', 'Phone Number']].head(10)

,Name,Email,Phone Number
0,Grace,email@example.com,NaN
1,David,user@domain.com,123-456-7890
2,Hannah,email@example.com,098-765-4321
3,Eve,name@company.org,
4,Grace,name@company.org,098-765-4321
5,Jack,user@domain.com,NaN
6,Charlie,NaN,123-456-7890
7,Grace,NaN,
8,Hannah,user@domain.com,123-456-7890
9,Eve,NaN,


- In Phone Number there are empty spaces and after strip those spaces , blank cells are appeared.

In [67]:
df['Phone Number'] = df['Phone Number'].replace('', np.nan)

In [68]:
df['Phone Number'].isna().sum()

np.int64(376)

In [70]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Name               1000 non-null   str           
 1   Age                1000 non-null   int64         
 2   Salary             1000 non-null   int64         
 3   Gender             1000 non-null   category      
 4   Department         1000 non-null   category      
 5   Position           1000 non-null   category      
 6   Joining Date       1000 non-null   datetime64[us]
 7   Performance Score  1000 non-null   category      
 8   Email              610 non-null    str           
 9   Phone Number       624 non-null    str           
dtypes: category(4), datetime64[us](1), int64(2), str(3)
memory usage: 73.0 KB


Name                   0
Age                    0
Salary                 0
Gender                 0
Department             0
Position               0
Joining Date           0
Performance Score      0
Email                390
Phone Number         376
dtype: int64

## Saving Cleaned Dataset

In [71]:
df.to_csv('HR_cleaned.csv', index=False)

## ✅ HR Dataset Cleaning Report

**Dataset:** HR Dataset | **Shape:** 1000 rows × 10 columns  
**Cleaned by:** Ume Habiba | **Tool:** Python, Pandas

---

### 🔍 Problems Found During Inspection

| Column | Issue |
|---|---|
| Age | str dtype, 335 word-form values, 159 nulls |
| Salary | str dtype, 310 word-form values, 167 hidden nulls (' NAN ') |
| Joining Date | mixed date formats |
| Phone Number | 185 nulls + 191 hidden nulls (blank spaces) |
| Email | 390 nulls |
| All Columns | incorrect dtypes |

---

### 🧹 Cleaning Steps Performed

#### Step 1: Word to Number Conversion
- Converted word-form values in **Age** and **Salary** using `word2number` library
- Used `try-except` to handle invalid strings gracefully

#### Step 2: Dtype Conversion
- Age, Salary → `int64` (after null handling)
- Gender, Department, Position, Performance Score → `category`
- Joining Date → `datetime64`

#### Step 3: Null Handling
| Column | Action | Reason |
|---|---|---|
| Age | Position-wise mean | Age varies by position |
| Salary | Department-wise mean | Pay scale varies by department |
| Email | Left as NaN | Cannot be guessed |
| Phone Number | Left as NaN | Cannot be guessed |

#### Step 4: Formatting & Standardization
- **Name** → strip whitespace + title case
- **Email** → strip whitespace + lowercase
- **Phone Number** → strip whitespace + empty strings → NaN

---

### 📊 Final Dataset Status

| Column | Dtype | Nulls | Status |
|---|---|---|---|
| Name | str | 0 | ✅ Clean |
| Age | int64 | 0 | ✅ Clean |
| Salary | int64 | 0 | ✅ Clean |
| Gender | category | 0 | ✅ Clean |
| Department | category | 0 | ✅ Clean |
| Position | category | 0 | ✅ Clean |
| Joining Date | datetime64 | 0 | ✅ Clean |
| Performance Score | category | 0 | ✅ Clean |
| Email | str | 390 | ✅ Intentional |
| Phone Number | str | 376 | ✅ Intentional |

---

### 💡 Key Decisions Made
- Word-form values **converted not dropped** — to avoid data loss
- Email and Phone nulls **intentionally kept** — contact still possible via other channel
- Age filled by **Position-wise mean** — age varies by seniority
- Salary filled by **Department-wise mean** — pay scale varies by department

---

### 🎯 Dataset is Clean and Ready for EDA!